# K-Nearest Neighbor (KNN) - Day 3 (From Scratch)

## Working of KNN Algorithm

The KNN algorithm operates on the principle of similarity where it predicts the label or value of a new data point by considering the labels or values of its K nearest neighbors in the training dataset.

---

## Step-by-Step Working

### Step 1: Selecting the optimal value of K

K is the number of nearest neighbors considered for prediction. The choice of K is important for model performance.

### Step 2: Calculating distance

To measure the similarity between target and training data points, Euclidean distance is widely used. Distance is calculated between data points in the dataset and target point.

### Step 3: Finding Nearest Neighbors

The K data points with the smallest distances to the target point are nearest neighbors.

### Step 4: Voting for Classification or Taking Average for Regression

**Classification (Majority Voting):**
- KNN looks at the K closest points
- Identifies which category the neighbors belong to
- Picks the category that appears the most

**Regression (Average):**
- KNN looks for the K closest points
- Takes the average of the values of those K neighbors
- This average is the predicted value

---

## Visualization of KNN Working

As the test point moves, the algorithm identifies the closest 'k' data points and assigns the test point the majority class label.

---

## One-Line Summary

**KNN works by finding K closest training points, then using majority vote (classification) or average (regression) to make predictions.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

print("="*50)
print("KNN DAY 3 - WORKING OF KNN ALGORITHM")
print("="*50)

## KNN Implementation with Step-by-Step Visualization

In [ ]:
class KNNVisual:
    """KNN with visualization of each step"""
    
    def __init__(self, k=5):
        self.k = k
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        print(f"Stored {len(self.X_train)} training samples")
    
    def _euclidean_distance(self, x1, x2):
        return np.sqrt(np.sum((x1 - x2) ** 2))
    
    def predict_with_steps(self, x, visualize=True):
        """
        Predict with step-by-step visualization
        """
        print("\n" + "="*50)
        print(f"PREDICTING FOR POINT: {x}")
        print("="*50)
        
        # Step 1: Calculate distances
        print("\nStep 1: Calculating distances to all training points")
        distances = []
        for i, xt in enumerate(self.X_train):
            dist = self._euclidean_distance(x, xt)
            distances.append((i, dist, self.y_train[i]))
            print(f"  Point {xt} (Class {self.y_train[i]}): Distance = {dist:.4f}")
        
        # Step 2: Sort by distance
        print("\nStep 2: Sorting by distance")
        distances.sort(key=lambda d: d[1])
        
        # Step 3: Get K nearest neighbors
        print(f"\nStep 3: Finding {self.k} nearest neighbors")
        neighbors = distances[:self.k]
        neighbor_labels = []
        for i, (idx, dist, label) in enumerate(neighbors):
            print(f"  Neighbor {i+1}: Point {self.X_train[idx]}, Distance: {dist:.4f}, Class: {label}")
            neighbor_labels.append(label)
        
        # Step 4: Majority voting
        print("\nStep 4: Majority voting")
        counter = Counter(neighbor_labels)
        print(f"  Votes: {dict(counter)}")
        prediction = counter.most_common(1)[0][0]
        print(f"  Prediction: Class {prediction}")
        
        if visualize:
            self._plot_prediction(x, neighbors)
        
        return prediction
    
    def _plot_prediction(self, x, neighbors):
        """Plot the prediction process"""
        plt.figure(figsize=(10, 8))
        
        # Plot all training points
        colors = ['red' if label == 0 else 'blue' for label in self.y_train]
        plt.scatter(self.X_train[:, 0], self.X_train[:, 1], 
                   c=colors, s=80, edgecolors='black', alpha=0.6, label='Training Points')
        
        # Plot target point
        plt.scatter(x[0], x[1], c='green', s=200, edgecolors='black', 
                   marker='*', label='Test Point')
        
        # Plot neighbors
        for idx, dist, label in neighbors:
            neighbor_point = self.X_train[idx]
            color = 'red' if label == 0 else 'blue'
            plt.scatter(neighbor_point[0], neighbor_point[1], 
                       c=color, s=150, edgecolors='yellow', linewidth=3, 
                       label='Neighbors' if idx == neighbors[0][0] else "")
            # Draw line from test point to neighbor
            plt.plot([x[0], neighbor_point[0]], [x[1], neighbor_point[1]], 
                    'gray', linestyle='--', alpha=0.5)
        
        plt.xlabel('Feature 1')
        plt.ylabel('Feature 2')
        plt.title(f'KNN Prediction (k={self.k})')
        plt.legend()
        plt.grid(alpha=0.3)
        plt.show()
    
    def predict(self, X):
        X = np.array(X)
        predictions = []
        for x in X:
            distances = [self._euclidean_distance(x, xt) for xt in self.X_train]
            k_indices = np.argsort(distances)[:self.k]
            k_labels = self.y_train[k_indices]
            predictions.append(Counter(k_labels).most_common(1)[0][0])
        return np.array(predictions)
    
    def score(self, X, y):
        return np.mean(self.predict(X) == np.array(y))

In [ ]:
# Create sample dataset
np.random.seed(42)

# Class 0 points
X0 = np.random.randn(20, 2) + np.array([-1, -1])
# Class 1 points
X1 = np.random.randn(20, 2) + np.array([1, 1])

X_train = np.vstack([X0, X1])
y_train = np.hstack([np.zeros(20), np.ones(20)])

print("Dataset created:")
print(f"Class 0: {np.sum(y_train == 0)} points")
print(f"Class 1: {np.sum(y_train == 1)} points")

In [ ]:
# Train KNN
knn = KNNVisual(k=5)
knn.fit(X_train, y_train)

In [ ]:
# Test point
test_point = np.array([0.5, 0.5])

# Predict with step-by-step visualization
prediction = knn.predict_with_steps(test_point)

In [ ]:
# Test multiple points
test_points = np.array([[-1.5, -1.5], [0, 0], [1.5, 1.5], [2, 2]])

print("\n" + "="*50)
print("Predictions for Multiple Test Points")
print("="*50)

for i, point in enumerate(test_points):
    pred = knn.predict([point])[0]
    label = "Class 0" if pred == 0 else "Class 1"
    print(f"Point {point} -> {label}")

In [ ]:
# Visualize decision boundary
def plot_decision_boundary(knn, X, y):
    plt.figure(figsize=(10, 8))
    
    # Plot points
    colors = ['red' if label == 0 else 'blue' for label in y]
    plt.scatter(X[:, 0], X[:, 1], c=colors, s=50, edgecolors='black')
    
    # Decision boundary
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05),
                         np.arange(y_min, y_max, 0.05))
    
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.contourf(xx, yy, Z, alpha=0.2, colors=['red', 'blue'])
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.title(f"KNN Decision Boundary (k={knn.k})")
    plt.grid(alpha=0.3)
    plt.show()

plot_decision_boundary(knn, X_train, y_train)

In [ ]:
# Day 3 Completed
print("\n" + "="*50)
print("KNN DAY 3 COMPLETED")
print("="*50)
print("Topics covered:")
print("- Step 1: Selecting optimal K")
print("- Step 2: Calculating distances")
print("- Step 3: Finding nearest neighbors")
print("- Step 4: Voting (classification) / Average (regression)")
print("- Step-by-step prediction visualization")
print("- Decision boundary visualization")
print("="*50)"
print("\n*** KNN MODULE COMPLETED ***")
print("\nComplete KNN Series:")
print("Day 1: Introduction and KNN from Scratch")
print("Day 2: Selecting K and Distance Metrics")
print("Day 3: Working of KNN Algorithm (Final)")